# Dense Mapped Logit-KD vs optimized matched CE: paired bootstrap gate

So sánh có cặp theo đúng 5.000 `image_id` Validation giữa Dense Mapped Logit-KD và CE-only continuation đã loại bỏ hoàn toàn Teacher, pseudo decoder, KL và mapping. Chạy sau khi cả ba optimized controls hoàn tất. Held-out Test luôn `SEALED / NOT USED`.


## Cell 1 — Integrity, contracts và source runs


In [1]:
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

STARTED = time.perf_counter()
SEEDS = (42, 123, 2026)
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
SEARCH_ROOT = INPUT_ROOT
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//"
    "XLKD05_Dense_Mapped_LogitKD_optimized_matched_ce_paired_bootstrap",
))
AUDIT = OUT / "audits"
RESULT = OUT / "results"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, RESULT, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(8 * 1024 * 1024), b""
        ):
            digest.update(block)
    return digest.hexdigest()

def inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, relative_path, target)
    return target

discovery = {}
for candidate in SEARCH_ROOT.rglob("*"):
    if candidate.is_file():
        discovery.setdefault(candidate.name, []).append(
            candidate.resolve()
        )

def exactly_one(name):
    matches = sorted(discovery.get(name, []))
    assert len(matches) == 1, (name, matches)
    return matches[0]

NEXT_PATH = exactly_one(
    "XLKD05_Dense_Mapped_LogitKD_"
    "NEXT_PHASE_ENTRY_COMPLETE.json"
)
NEXT = read_json(NEXT_PATH)
assert NEXT["status"] == "complete"
assert NEXT["reproduction_gate"] == "passed"
assert NEXT["final_scientific_gate"] == "pending"
assert NEXT["test_set_status"] == "sealed"
assert NEXT["test_set_used"] is False

contract_ref = NEXT["paired_bootstrap_contract"]
CONTRACT_PATH = inside(
    NEXT_PATH.parent, contract_ref["relative_path"]
)
assert sha256(CONTRACT_PATH) == contract_ref["sha256"]
CONTRACT = read_json(CONTRACT_PATH)
assert CONTRACT["status"] == "locked"
assert int(CONTRACT["validation_image_count"]) == 5000
assert int(CONTRACT["replicates"]) == 10000
assert int(CONTRACT["bootstrap_seed"]) == 20261018
assert CONTRACT["primary_metric"] == "CIDEr-D"
assert CONTRACT["test_set_status"] == "sealed"
assert CONTRACT["test_set_used"] is False

kd_names = {
    42: "XLKD04_DENSE_MAPPED_LOGIT_KD_SEED42_FULL_RUN_COMPLETE.json",
    123: "XLKD05_DENSE_MAPPED_LOGIT_KD_SEED123_FULL_RUN_COMPLETE.json",
    2026: "XLKD05_DENSE_MAPPED_LOGIT_KD_SEED2026_FULL_RUN_COMPLETE.json",
}
runs = {}
for seed in SEEDS:
    kd_path = exactly_one(kd_names[seed])
    control_path = exactly_one(
        f"OPTIMIZED_MATCHED_CE_SEED{seed}_FULL_RUN_COMPLETE.json"
    )
    kd = read_json(kd_path)
    control = read_json(control_path)
    assert kd["status"] == control["status"] == "complete"
    assert int(kd["seed"]) == int(control["seed"]) == seed
    assert kd["selection_split"] == control["selection_split"] == (
        "validation"
    )
    assert kd["test_set_status"] == control["test_set_status"] == (
        "sealed"
    )
    assert kd["test_set_used"] is False
    assert control["test_set_used"] is False
    assert control["matched_budget_control"] is True
    assert control["optimized_ce_engine"] is True
    assert control["teacher_loaded"] is False
    assert control["pseudo_decoder_forward_calls"] == 0
    assert control["kl_calls"] == 0
    assert control["mapping_operations"] == 0
    assert control["independent_early_stopping_used"] is False
    assert control["ready_for_paired_bootstrap"] is True
    assert float(control["alpha_human_ce"]) == 1.0
    assert float(control["alpha_kd"]) == 0.0

    kd_epoch = int(kd["best_result"]["epoch"])
    control_epoch = int(control["best_result"]["epoch"])
    kd_predictions = (
        kd_path.parent / "validation"
        / f"epoch_{kd_epoch:02d}_predictions.json"
    )
    control_predictions = (
        control_path.parent / "validation"
        / f"epoch_{control_epoch:02d}_predictions.json"
    )
    references = (
        kd_path.parent / "validation"
        / "karpathy_validation_references.json"
    )
    for path in [kd_predictions, control_predictions, references]:
        assert path.is_file(), path
    runs[seed] = {
        "kd_path": kd_path,
        "control_path": control_path,
        "kd": kd,
        "control": control,
        "kd_predictions": kd_predictions,
        "control_predictions": control_predictions,
        "references": references,
    }

write_json(AUDIT / "cell01_inputs.json", {
    "schema_version": "xlkd05_bootstrap_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "seeds": list(SEEDS),
    "kd_runs": "3 / 3",
    "optimized_matched_ce_runs": "3 / 3",
    "zero_kd_computation_verified": True,
    "contract_sha256": sha256(CONTRACT_PATH),
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
})

print("Input integrity            : PASSED")
print("KD / matched CE runs      : 3 / 3 | 3 / 3")
print("Bootstrap contract        : LOCKED")
print("Held-out Test             : SEALED")


Input integrity            : PASSED
KD / matched CE runs      : 3 / 3 | 3 / 3
Bootstrap contract        : LOCKED
Held-out Test             : SEALED


## Cell 2 — COCO scorers và image-ID pairing


In [2]:
try:
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.cider.cider import Cider
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.tokenizer.ptbtokenizer import (
        PTBTokenizer,
    )
except ImportError:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl"))
    assert len(wheels) >= 1, (
        "Thiếu pycocoevalcap và không tìm thấy offline wheel."
    )
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-index", "--no-deps", str(wheels[0]),
    ])
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.cider.cider import Cider
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.tokenizer.ptbtokenizer import (
        PTBTokenizer,
    )

# pycocoevalcap 1.2 wheels in some offline environments place
# ``-Xmx2G`` after ``-jar``. Java then treats it as the JAR filename
# and the METEOR subprocess exits on the first request. Override only
# the constructor so the official scorer protocol remains unchanged.
import pycocoevalcap.meteor.meteor as meteor_module

def fixed_meteor_init(self):
    meteor_source = Path(
        meteor_module.__file__
    ).resolve().parent
    meteor_runtime = meteor_source
    # METEOR 1.5 double-encodes spaces while resolving its bundled
    # paraphrase table. Kaggle paths do not contain spaces, but this
    # copy makes local Windows verification equally deterministic.
    if " " in str(meteor_source):
        meteor_runtime = OUT / "_meteor_runtime"
        shutil.copytree(
            meteor_source,
            meteor_runtime,
            dirs_exist_ok=True,
        )
    meteor_jar = meteor_runtime / "meteor-1.5.jar"
    self.meteor_cmd = [
        "java",
        "-Xmx2G",
        "-jar",
        str(meteor_jar),
        "-",
        "-",
        "-stdio",
        "-l",
        "en",
        "-norm",
    ]
    self.meteor_p = subprocess.Popen(
        self.meteor_cmd,
        cwd=meteor_runtime,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    self.lock = threading.Lock()

Meteor.__init__ = fixed_meteor_init

def coco_payload(references, predictions, image_ids):
    raw_gts = {
        image_id: [
            {"caption": caption}
            for caption in references[str(image_id)]
        ]
        for image_id in image_ids
    }
    raw_res = {
        image_id: [{
            "caption": predictions[str(image_id)]["caption"]
        }]
        for image_id in image_ids
    }
    tokenizer = PTBTokenizer()
    return (
        tokenizer.tokenize(raw_gts),
        tokenizer.tokenize(raw_res),
    )

def score_per_image(gts, res):
    output = {}
    bleu_score, bleu_scores = Bleu(4).compute_score(gts, res)
    output["BLEU-1"] = np.asarray(
        bleu_scores[0], dtype=np.float64
    )
    output["BLEU-4"] = np.asarray(
        bleu_scores[3], dtype=np.float64
    )
    meteor = Meteor()
    try:
        _, meteor_scores = meteor.compute_score(gts, res)
    finally:
        if hasattr(meteor, "close"):
            meteor.close()
    output["METEOR"] = np.asarray(
        meteor_scores, dtype=np.float64
    )
    _, rouge_scores = Rouge().compute_score(gts, res)
    output["ROUGE-L"] = np.asarray(
        rouge_scores, dtype=np.float64
    )
    _, cider_scores = Cider().compute_score(gts, res)
    output["CIDEr-D"] = np.asarray(
        cider_scores, dtype=np.float64
    )
    return output

SCORES = {}
pairing_audits = []
for seed in SEEDS:
    run = runs[seed]
    references = read_json(run["references"])
    kd_predictions = read_json(run["kd_predictions"])
    control_predictions = read_json(run["control_predictions"])
    reference_ids = sorted(map(int, references))
    kd_ids = sorted(map(int, kd_predictions))
    control_ids = sorted(map(int, control_predictions))
    assert len(reference_ids) == 5000
    assert reference_ids == kd_ids == control_ids

    kd_gts, kd_res = coco_payload(
        references, kd_predictions, reference_ids
    )
    control_gts, control_res = coco_payload(
        references, control_predictions, reference_ids
    )
    kd_scores = score_per_image(kd_gts, kd_res)
    control_scores = score_per_image(control_gts, control_res)
    for metric in CONTRACT["secondary_metrics"] + ["CIDEr-D"]:
        assert kd_scores[metric].shape == (5000,)
        assert control_scores[metric].shape == (5000,)
        assert np.isfinite(kd_scores[metric]).all()
        assert np.isfinite(control_scores[metric]).all()
    # CIDEr-D is the mean of the image-level scores returned by the
    # official scorer. Reproducing the packaged best-epoch values
    # detects tokenization, reference or prediction mismatches before
    # any bootstrap decision is made.
    expected_kd_cider = float(
        run["kd"]["best_result"]["metrics"]["CIDEr-D"]
    )
    expected_control_cider = float(
        run["control"]["best_result"]["metrics"]["CIDEr-D"]
    )
    observed_kd_cider = float(kd_scores["CIDEr-D"].mean())
    observed_control_cider = float(
        control_scores["CIDEr-D"].mean()
    )
    assert np.isclose(
        observed_kd_cider,
        expected_kd_cider,
        rtol=0.0,
        atol=1e-6,
    ), (
        seed,
        "KD CIDEr-D reproduction mismatch",
        observed_kd_cider,
        expected_kd_cider,
    )
    assert np.isclose(
        observed_control_cider,
        expected_control_cider,
        rtol=0.0,
        atol=1e-6,
    ), (
        seed,
        "control CIDEr-D reproduction mismatch",
        observed_control_cider,
        expected_control_cider,
    )
    SCORES[seed] = {
        "image_ids": np.asarray(reference_ids, dtype=np.int64),
        "kd": kd_scores,
        "control": control_scores,
    }
    pairing_audits.append({
        "seed": seed,
        "image_count": len(reference_ids),
        "image_id_sha256": hashlib.sha256(
            np.asarray(
                reference_ids, dtype=np.int64
            ).tobytes()
        ).hexdigest(),
        "exact_pairing": True,
        "kd_cider_reproduced": observed_kd_cider,
        "control_cider_reproduced": observed_control_cider,
    })

write_json(AUDIT / "cell02_pairing.json", {
    "schema_version": "xlkd05_bootstrap_cell02_v1",
    "created_at_utc": now(),
    "status": "passed",
    "pairing": pairing_audits,
    "finite_scores": True,
    "test_set_status": "sealed",
    "test_set_used": False,
})

print("Image-ID pairing          : 3 / 3 EXACT")
print("Images per seed           : 5,000")
print("Per-image scores          : FINITE")
print("Held-out Test             : SEALED")


Processing /kaggle/input/datasets/nguynthnhdan/pycocoevalcap-1-2/pycocoevalcap-1.2-py3-none-any.whl


PTBTokenizer tokenized 307820 tokens at 666954.99 tokens per second.
PTBTokenizer tokenized 52207 tokens at 274046.42 tokens per second.
PTBTokenizer tokenized 307820 tokens at 664129.81 tokens per second.
PTBTokenizer tokenized 54357 tokens at 249012.90 tokens per second.


{'testlen': 47208, 'reflen': 47276, 'guess': [47208, 42208, 37208, 32208], 'correct': [35838, 19867, 9622, 4485]}
ratio: 0.9985616380404223
{'testlen': 49358, 'reflen': 48635, 'guess': [49358, 44358, 39358, 34358], 'correct': [35314, 18523, 9139, 4564]}
ratio: 1.014865837359905


PTBTokenizer tokenized 307820 tokens at 638732.91 tokens per second.
PTBTokenizer tokenized 52160 tokens at 269936.87 tokens per second.
PTBTokenizer tokenized 307820 tokens at 692418.83 tokens per second.
PTBTokenizer tokenized 54061 tokens at 272284.44 tokens per second.


{'testlen': 47161, 'reflen': 47175, 'guess': [47161, 42161, 37161, 32161], 'correct': [35820, 19926, 9691, 4555]}
ratio: 0.999703232644388
{'testlen': 49062, 'reflen': 48445, 'guess': [49062, 44062, 39062, 34062], 'correct': [35222, 18562, 9195, 4575]}
ratio: 1.012736092475983


PTBTokenizer tokenized 307820 tokens at 608625.46 tokens per second.
PTBTokenizer tokenized 52190 tokens at 233356.78 tokens per second.
PTBTokenizer tokenized 307820 tokens at 662623.62 tokens per second.
PTBTokenizer tokenized 54488 tokens at 255005.40 tokens per second.


{'testlen': 47191, 'reflen': 47263, 'guess': [47191, 42191, 37191, 32191], 'correct': [35773, 19889, 9721, 4561]}
ratio: 0.9984766096100334
{'testlen': 49489, 'reflen': 48770, 'guess': [49489, 44489, 39489, 34489], 'correct': [35323, 18520, 9126, 4595]}
ratio: 1.0147426696739592
Image-ID pairing          : 3 / 3 EXACT
Images per seed           : 5,000
Per-image scores          : FINITE
Held-out Test             : SEALED


## Cell 3 — 10.000 paired bootstrap replicates


In [3]:
METRICS = tuple(CONTRACT["secondary_metrics"]) + ("CIDEr-D",)
REPLICATES = int(CONTRACT["replicates"])
BOOTSTRAP_SEED = int(CONTRACT["bootstrap_seed"])

def paired_bootstrap(metric_differences, rng, replicates):
    metric_differences = np.asarray(
        metric_differences, dtype=np.float64
    )
    n = metric_differences.shape[0]
    values = np.empty(replicates, dtype=np.float64)
    chunk = 100
    cursor = 0
    while cursor < replicates:
        count = min(chunk, replicates - cursor)
        indices = rng.integers(
            0, n, size=(count, n), endpoint=False
        )
        values[cursor:cursor + count] = (
            metric_differences[indices].mean(axis=1)
        )
        cursor += count
    return {
        "mean_delta": float(metric_differences.mean()),
        "bootstrap_mean": float(values.mean()),
        "lower_95": float(np.quantile(values, 0.025)),
        "upper_95": float(np.quantile(values, 0.975)),
        "replicates": replicates,
    }

rows = []
detailed = {}
for seed in SEEDS:
    rng = np.random.default_rng(BOOTSTRAP_SEED + seed)
    detailed[str(seed)] = {}
    for metric in METRICS:
        differences = (
            SCORES[seed]["kd"][metric]
            - SCORES[seed]["control"][metric]
        )
        result = paired_bootstrap(
            differences, rng, REPLICATES
        )
        result["gate_positive_lower_95"] = (
            result["mean_delta"] > 0.0
            and result["lower_95"] > 0.0
        )
        detailed[str(seed)][metric] = result
        rows.append({
            "seed": seed,
            "metric": metric,
            **result,
        })

RESULT_JSON = RESULT / "paired_bootstrap_results.json"
RESULT_CSV = RESULT / "paired_bootstrap_results.csv"
write_json(RESULT_JSON, {
    "schema_version": "xlkd05_paired_bootstrap_results_v1",
    "created_at_utc": now(),
    "seeds": list(SEEDS),
    "unit": "image_id",
    "validation_image_count": 5000,
    "results": detailed,
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
})
with RESULT_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(rows[0])
    )
    writer.writeheader()
    writer.writerows(rows)

write_json(AUDIT / "cell03_bootstrap.json", {
    "schema_version": "xlkd05_bootstrap_cell03_v1",
    "created_at_utc": now(),
    "status": "passed",
    "replicates_per_seed_metric": REPLICATES,
    "metrics": list(METRICS),
    "result_sha256": sha256(RESULT_JSON),
    "finite_results": all(
        np.isfinite([
            row["mean_delta"], row["lower_95"],
            row["upper_95"]
        ]).all()
        for row in rows
    ),
    "test_set_status": "sealed",
    "test_set_used": False,
})

for seed in SEEDS:
    cider = detailed[str(seed)]["CIDEr-D"]
    print(
        f"seed={seed} | delta CIDEr-D="
        f"{cider['mean_delta']:+.6f} | "
        f"95% CI=[{cider['lower_95']:+.6f}, "
        f"{cider['upper_95']:+.6f}]"
    )


seed=42 | delta CIDEr-D=+0.090933 | 95% CI=[+0.075487, +0.106458]
seed=123 | delta CIDEr-D=+0.088763 | 95% CI=[+0.072924, +0.104643]
seed=2026 | delta CIDEr-D=+0.087640 | 95% CI=[+0.071222, +0.104052]


## Cell 4 — Final scientific gate và đóng gói


In [4]:
required_count = 2
passed_seeds = [
    seed for seed in SEEDS
    if detailed[str(seed)]["CIDEr-D"][
        "gate_positive_lower_95"
    ]
]
secondary_non_regression = all(
    detailed[str(seed)][metric]["mean_delta"] >= -0.002
    for seed in SEEDS
    for metric in ("BLEU-4", "METEOR", "ROUGE-L")
)
FINAL_GATE = (
    len(passed_seeds) >= required_count
    and secondary_non_regression
)

DECISION_PATH = RESULT / "final_scientific_gate.json"
write_json(DECISION_PATH, {
    "schema_version": "xlkd05_final_scientific_gate_v1",
    "created_at_utc": now(),
    "status": "passed" if FINAL_GATE else "failed",
    "primary_metric": "CIDEr-D",
    "passed_seeds": passed_seeds,
    "required_seed_count": "at least 2 / 3",
    "observed_seed_count": f"{len(passed_seeds)} / 3",
    "secondary_non_regression": secondary_non_regression,
    "matched_training_budget": True,
    "paired_image_level_bootstrap": True,
    "posthoc_threshold_relaxation": False,
    "decision": (
        "Dense Mapped Logit-KD final claim authorized"
        if FINAL_GATE
        else "Dense Mapped Logit-KD final claim not supported"
    ),
    "held_out_test_may_be_opened": False,
    "next": (
        "Complete Feature KD preflight/screening and lock one final "
        "method before opening Test once."
    ),
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
})

artifact_paths = sorted(
    [
        path for path in OUT.rglob("*")
        if path.is_file()
        and not path.is_relative_to(MANIFEST)
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUT).as_posix(),
)
rows = [
    {
        "relative_path": path.relative_to(OUT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    }
    for path in artifact_paths
]
MANIFEST_CSV = MANIFEST / "paired_bootstrap_manifest.csv"
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(rows)

MARKER = OUT / (
    "XLKD05_Dense_Mapped_LogitKD_"
    "OPTIMIZED_MATCHED_CE_PAIRED_BOOTSTRAP_COMPLETE.json"
)
write_json(MARKER, {
    "schema_version": "xlkd05_paired_bootstrap_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "final_scientific_gate": (
        "passed" if FINAL_GATE else "failed"
    ),
    "passed_seeds": passed_seeds,
    "optimized_matched_budget_controls_verified": "3 / 3",
    "zero_kd_computation_verified": True,
    "bootstrap_replicates": REPLICATES,
    "decision": {
        "relative_path": DECISION_PATH.relative_to(OUT).as_posix(),
        "sha256": sha256(DECISION_PATH),
    },
    "results": {
        "relative_path": RESULT_JSON.relative_to(OUT).as_posix(),
        "sha256": sha256(RESULT_JSON),
    },
    "manifest": {
        "relative_path": MANIFEST_CSV.relative_to(
            OUT
        ).as_posix(),
        "sha256": sha256(MANIFEST_CSV),
        "rows": len(rows),
    },
    "held_out_test_may_be_opened": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("=" * 78)
print("PAIRED BOOTSTRAP FINAL GATE")
print("=" * 78)
print(
    "Final scientific gate     :",
    "PASSED" if FINAL_GATE else "FAILED",
)
print(
    "Positive lower 95% CI     :",
    f"{len(passed_seeds)} / 3 seeds",
)
print(
    "Secondary non-regression  :",
    "PASSED" if secondary_non_regression else "FAILED",
)
print("Held-out Test             : SEALED")
print("Marker                    :", MARKER)


PAIRED BOOTSTRAP FINAL GATE
Final scientific gate     : PASSED
Positive lower 95% CI     : 3 / 3 seeds
Secondary non-regression  : PASSED
Held-out Test             : SEALED
Marker                    : /kaggle/working//XLKD05_Dense_Mapped_LogitKD_optimized_matched_ce_paired_bootstrap/XLKD05_Dense_Mapped_LogitKD_OPTIMIZED_MATCHED_CE_PAIRED_BOOTSTRAP_COMPLETE.json
